# PySpark reference 10: reading and writing data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-10-reading-and-writing.ipynb)

Readers and their options, schemas, corrupt records, file metadata and file selection, JDBC, writers and save modes, partitioning and dynamic overwrite, file sizing, tables and bucketing, and Delta Lake: history, time travel, MERGE, replaceWhere, schema evolution and maintenance. Practise it in [PySpark practice 10: reading and writing data](pyspark-practice-10-reading-and-writing.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. The session loads Delta Lake (its jars download on first use); `files_in(path)` lists the data files a write produced and `write_text(path, text)` drops a raw file.

In [ ]:
import os
import sys
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the examples write

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import glob
import os


def files_in(path, pattern="*.parquet"):
    """Data files under path (recursively), for showing what a write produced."""
    return sorted(os.path.relpath(p, path) for p in glob.glob(os.path.join(path, "**", pattern), recursive=True))


def write_text(path, text):
    """Write a small raw file, the way a source system would drop it."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text)

## Contents

| section | entries |
| --- | --- |
| [Reading data](#reading-data) | [`spark.read`: `DataFrameReader`](#sparkread-dataframereader), [CSV options](#csv-options), [JSON, Parquet and ORC](#json-parquet-and-orc), [Malformed records: `mode` and `_corrupt_record`](#malformed-records-mode-and-_corrupt_record), [File metadata: the `_metadata` column](#file-metadata-the-_metadata-column), [Selecting files: globs, `pathGlobFilter`, `recursiveFileLookup`, `modifiedAfter`](#selecting-files-globs-pathglobfilter-recursivefilelookup-modifiedafter), [JDBC sources](#jdbc-sources) |
| [Writing data](#writing-data) | [`df.write`: `DataFrameWriter`](#dfwrite-dataframewriter), [`partitionBy` and partition pruning](#partitionby-and-partition-pruning), [Dynamic partition overwrite](#dynamic-partition-overwrite), [Controlling file count and size](#controlling-file-count-and-size), [Tables: `saveAsTable` and `insertInto`](#tables-saveastable-and-insertinto), [Bucketing: `bucketBy` and `sortBy`](#bucketing-bucketby-and-sortby) |
| [Delta Lake](#delta-lake) | [Writing and reading Delta tables, history and time travel](#writing-and-reading-delta-tables-history-and-time-travel), [`MERGE`: upserts with `DeltaTable.merge`](#merge-upserts-with-deltatablemerge), [Replace part of a table: `replaceWhere`](#replace-part-of-a-table-replacewhere), [Schema enforcement and evolution](#schema-enforcement-and-evolution), [Maintenance: `OPTIMIZE`, `ZORDER BY` and `VACUUM`](#maintenance-optimize-zorder-by-and-vacuum) |

## Reading data

### `spark.read`: `DataFrameReader`

`spark.read` returns a `DataFrameReader`. Configure it, then call a format method or `load`. Reading is lazy for most
formats except schema inference, which scans data immediately.

| method | description |
| --- | --- |
| `.format(name)` | `"parquet"` (default), `"csv"`, `"json"`, `"orc"`, `"text"`, `"delta"`, `"jdbc"`, `"avro"` (with the spark-avro package), ... |
| `.schema(schema)` | a DDL string or `StructType`; skips inference and fixes the types |
| `.option(key, value)` / `.options(**kw)` | format-specific settings (tables below) |
| `.load(path or paths)` | read with the configured format |
| `.csv(path, **opts)`, `.json(...)`, `.parquet(...)`, `.orc(...)`, `.text(...)`, `.table(name)` | shortcuts |

**Returns:** a DataFrame.

**Notes**
- Always pass an explicit schema for CSV and JSON in pipelines: inference scans the data (slow), guesses types from
  a sample, and changes when the data changes.
- Paths accept globs (`/data/2026/01/*/*.csv`), lists of paths and directories.

In [ ]:
raw_dir = f"{WORK_DIR}/raw"
write_text(f"{raw_dir}/orders.csv", "order_id;amount;order_date;note\n1;120.50;05/01/2026;first\n2;-;06/01/2026;\n3;99.00;07/01/2026;\"semi;colon\"\n")
inferred = spark.read.option("header", True).option("sep", ";").option("inferSchema", True).csv(f"{raw_dir}/orders.csv")
print("inferred:", inferred.dtypes)
typed = (spark.read.schema("order_id int, amount decimal(10,2), order_date date, note string")
         .options(header=True, sep=";", nullValue="-", dateFormat="dd/MM/yyyy")
         .csv(f"{raw_dir}/orders.csv"))
typed.show()

### CSV options

| option | default | meaning |
| --- | --- | --- |
| `header` | false | first line holds column names |
| `sep` / `delimiter` | `,` | field separator |
| `quote` / `escape` | `"` / `\` | quoting and escaping inside fields |
| `nullValue` | empty string | text that means null (`-`, `NULL`, `N/A`) |
| `emptyValue` | empty string | text used for empty strings |
| `dateFormat` / `timestampFormat` | ISO | patterns for date and timestamp columns |
| `inferSchema` | false | scan the data to guess types |
| `enforceSchema` | true | apply the given schema even if the header disagrees |
| `multiLine` | false | allow quoted fields with line breaks |
| `encoding` | UTF-8 | file encoding |
| `ignoreLeadingWhiteSpace` / `ignoreTrailingWhiteSpace` | false | trim unquoted values |
| `mode` | `PERMISSIVE` | how to handle malformed rows ([corrupt records](#malformed-records-mode-and-_corrupt_record)) |

**Notes:** a header row does not set types; with `inferSchema` off every column is a string.

### JSON, Parquet and ORC

| format | key options | notes |
| --- | --- | --- |
| JSON | `multiLine` (one document spanning lines, or a JSON array per file), `primitivesAsString`, `allowComments`, `dateFormat`, `timestampFormat` | default is JSON Lines: one object per line |
| Parquet | `mergeSchema` (union the schemas of all files), `datetimeRebaseMode` | schema is stored in the files; supports predicate and column pushdown |
| ORC | `mergeSchema` | columnar like Parquet, common in Hive estates |

**Notes:** Parquet reads the schema from file footers, so no inference pass is needed. `mergeSchema` reads every
footer, which is slow on many files; enable it only when files really have different columns.

In [ ]:
write_text(f"{raw_dir}/events.jsonl", '{"id": 1, "type": "view"}\n{"id": 2, "type": "buy", "amount": 12.5}\n')
write_text(f"{raw_dir}/doc.json", '[{"id": 3, "type": "view"},\n {"id": 4, "type": "buy"}]')
spark.read.json(f"{raw_dir}/events.jsonl").show()
spark.read.option("multiLine", True).json(f"{raw_dir}/doc.json").show()

pq = f"{WORK_DIR}/evolving"
spark.createDataFrame([(1, "a")], "id int, code string").write.mode("overwrite").parquet(f"{pq}/part=1")
spark.createDataFrame([(2, "b", 9.5)], "id int, code string, score double").write.mode("overwrite").parquet(f"{pq}/part=2")
print("without mergeSchema:", spark.read.parquet(pq).columns)
print("with mergeSchema:   ", spark.read.option("mergeSchema", True).parquet(pq).columns)

### Malformed records: `mode` and `_corrupt_record`

What the CSV and JSON readers do with a row that does not fit the schema:

| `mode` | behaviour |
| --- | --- |
| `PERMISSIVE` (default) | bad fields become null; the whole raw line goes to the column named by `columnNameOfCorruptRecord` if that column is in the schema |
| `DROPMALFORMED` | bad rows are dropped silently |
| `FAILFAST` | the first bad row fails the read |

**Use it for:** a bronze layer that keeps every row and routes rejects to a quarantine table: read in `PERMISSIVE` mode
with a `_corrupt_record string` column in the schema, then split on whether it is null.

**Notes**
- Spark refuses a query that references only the corrupt-record column of a raw file scan; cache or select other
  columns too.
- CSV parsing is pruned to the columns a query needs, so `count()` does not parse fields and cannot see malformed
  values: `DROPMALFORMED` then appears to keep every row. Validate with a query that reads the columns.
- `DROPMALFORMED` hides data loss; prefer `PERMISSIVE` with a reject count.

In [ ]:
write_text(f"{raw_dir}/dirty.csv", "id,qty\n1,4\n2,x\n3,5,extra\n4,\n")
schema = "id int, qty int, _corrupt_record string"
permissive = spark.read.schema(schema).option("header", True).option("columnNameOfCorruptRecord", "_corrupt_record").csv(f"{raw_dir}/dirty.csv").cache()
permissive.show(truncate=False)
dropmal = spark.read.schema("id int, qty int").option("header", True).option("mode", "DROPMALFORMED").csv(f"{raw_dir}/dirty.csv")
print("DROPMALFORMED count():", dropmal.count(), "| rows actually returned:", len(dropmal.collect()))
with expected_error():
    spark.read.schema("id int, qty int").option("header", True).option("mode", "FAILFAST").csv(f"{raw_dir}/dirty.csv").collect()
_ = permissive.unpersist()

### File metadata: the `_metadata` column

File-based sources expose a hidden `_metadata` struct with `file_path`, `file_name`, `file_size`,
`file_block_start`, `file_block_length` and `file_modification_time`. Select it explicitly to record lineage.

**Use it for:** recording which file each row came from in bronze tables, and incremental reads by modification time.

**Notes:** it is not part of `*`; select `"_metadata.file_name"` (or the whole struct) explicitly.

In [ ]:
drop = f"{WORK_DIR}/drop"
orders.repartition(2).write.mode("overwrite").parquet(drop)
(spark.read.parquet(drop).select("order_id", F.substring("_metadata.file_name", 1, 10).alias("file"), "_metadata.file_size")
 .groupBy("file", "file_size").agg(F.count("*").alias("rows")).orderBy("file").show())

### Selecting files: globs, `pathGlobFilter`, `recursiveFileLookup`, `modifiedAfter`

| option | meaning |
| --- | --- |
| glob in the path | `/data/2026/01/*/orders_*.csv` |
| `pathGlobFilter` | keep only file names matching a pattern, e.g. `*.csv` |
| `recursiveFileLookup` | read files in all subdirectories (disables partition discovery) |
| `modifiedAfter` / `modifiedBefore` | only files modified in a time range (timestamp text) |

**Notes:** directories named `col=value` are discovered as partition columns automatically unless
`recursiveFileLookup` is on.

In [ ]:
landing = f"{WORK_DIR}/landing"
for day in ["2026-01-05", "2026-01-06"]:
    write_text(f"{landing}/day={day}/orders.csv", "order_id,amount\n1,10\n2,20\n")
    write_text(f"{landing}/day={day}/README.txt", "not data")
spark.read.option("header", True).option("pathGlobFilter", "*.csv").csv(landing).show()
print("recursive lookup columns:", spark.read.option("header", True).option("pathGlobFilter", "*.csv").option("recursiveFileLookup", True).csv(landing).columns)

### JDBC sources

Reading a database table in parallel needs a numeric, date or timestamp column to split on. Without
`partitionColumn`, the whole table is read by **one** task.

```python
(spark.read.format("jdbc")
 .option("url", "jdbc:postgresql://host:5432/shop")
 .option("dbtable", "public.orders")          # or .option("query", "select ... where ...")
 .option("user", user).option("password", password)
 .option("partitionColumn", "order_id")       # numeric, date or timestamp
 .option("lowerBound", 1).option("upperBound", 10_000_000)
 .option("numPartitions", 16)                 # also caps concurrent connections
 .option("fetchsize", 10_000)
 .load())
```

**Notes:** `lowerBound` and `upperBound` only set the split points, they do not filter rows. Filters and
column selections are pushed into the database query when possible (`pushDownPredicate`). The driver JAR must be on
the classpath (`spark.jars.packages`).

## Writing data

### `df.write`: `DataFrameWriter`

| method | description |
| --- | --- |
| `.format(name)` | output format (default `parquet`) |
| `.mode(saveMode)` | what to do if the target exists (below) |
| `.option(key, value)` | format settings: `compression`, `header`, `maxRecordsPerFile`, `replaceWhere`, ... |
| `.partitionBy(*cols)` | one directory per value: `col=value/` |
| `.bucketBy(n, *cols)`, `.sortBy(*cols)` | hash the rows into `n` files per partition (tables only) |
| `.save(path)` | write to a path |
| `.saveAsTable(name)` | write and register a table in the catalog |
| `.insertInto(name)` | insert into an existing table **by column position** |
| `.csv(path)`, `.json(path)`, `.parquet(path)`, ... | shortcuts |

| `mode` | if the target exists |
| --- | --- |
| `"errorifexists"` / `"error"` (default) | fail |
| `"append"` | add the new files |
| `"overwrite"` | replace everything (or only the written partitions with dynamic overwrite) |
| `"ignore"` | do nothing |

**Notes:** a write is an action. Each task writes its own files, so the number of output files equals the number of
partitions being written (times partition directories).

In [ ]:
out = f"{WORK_DIR}/orders_out"
orders.write.mode("overwrite").option("compression", "zstd").parquet(out)
print(files_in(out, "*.parquet"))
orders.limit(2).write.mode("append").parquet(out)
print("rows after append:", spark.read.parquet(out).count())
with expected_error():
    orders.write.parquet(out)  # default mode errorifexists

### `partitionBy` and partition pruning

`partitionBy("col")` writes one directory per distinct value (`status=paid/`). Readers that filter on the partition
column skip the other directories entirely (**partition pruning**), visible as `PartitionFilters` in the plan.

**Use it for:** low-cardinality columns that most queries filter on: dates, regions, statuses.

**Notes:** partitioning by a high-cardinality column (customer id, timestamp) creates thousands of tiny files and
directories; target partitions of at least several hundred MB. The partition column is removed from the data files
and restored from the path on read.

In [ ]:
by_status = f"{WORK_DIR}/orders_by_status"
orders.write.mode("overwrite").partitionBy("status").parquet(by_status)
print(sorted(os.listdir(by_status)))
paid = spark.read.parquet(by_status).where("status = 'paid'")
print([l.strip()[:90] for l in plan_lines(paid, "formatted") if "PartitionFilters" in l])

### Dynamic partition overwrite

By default, `mode("overwrite")` on a partitioned path deletes **every** partition and writes the new data. With
`spark.sql.sources.partitionOverwriteMode=dynamic`, only the partitions present in the new data are replaced; others are
left untouched.

**Use it for:** idempotent daily reloads: recompute one day and overwrite just that day's partition.

**Notes**
- Set it per write with `.option("partitionOverwriteMode", "dynamic")` rather than session-wide.
- Keep column types identical to the existing partitions. `amount * 2` widens `decimal(12,2)` to `decimal(13,2)`;
  a partition written with a different physical type makes later reads of the whole table fail with
  `PARQUET_COLUMN_DATA_TYPE_MISMATCH`. Cast back to the table's types before writing.
- Delta tables use `replaceWhere` instead ([Delta replace](#replace-part-of-a-table-replacewhere)), and enforce the schema for you.

In [ ]:
daily = orders.withColumn("day", F.to_date("order_ts"))
daily_path = f"{WORK_DIR}/orders_daily"
daily.write.mode("overwrite").partitionBy("day").parquet(daily_path)
fix = daily.where("day = '2026-01-05'").withColumn("amount", (F.col("amount") * 2).cast("decimal(12,2)"))  # keep the table's type
fix.write.mode("overwrite").option("partitionOverwriteMode", "dynamic").partitionBy("day").parquet(daily_path)
print("days kept:", spark.read.parquet(daily_path).select("day").distinct().count(), "| rows:", spark.read.parquet(daily_path).count())
fix.write.mode("overwrite").partitionBy("day").parquet(daily_path)  # static overwrite
print("after static overwrite, days:", spark.read.parquet(daily_path).select("day").distinct().count())

### Controlling file count and size

| tool | effect |
| --- | --- |
| `df.repartition(n)` before writing | `n` files (per partition directory, when combined with `partitionBy`, one file per task per directory) |
| `df.repartition("col").write.partitionBy("col")` | one file per partition directory |
| `df.coalesce(n)` | fewer files without a shuffle (reduces upstream parallelism too) |
| `.option("maxRecordsPerFile", n)` | split any file with more than `n` rows |
| AQE coalescing | merges small shuffle partitions before the write |

**Notes:** aim for files of roughly 128 MB–1 GB. Many small files slow every later read (one task and one metadata
request per file); compact them periodically (`OPTIMIZE` on Delta).

In [ ]:
sized = f"{WORK_DIR}/sized"
orders.coalesce(1).write.mode("overwrite").option("maxRecordsPerFile", 5).parquet(sized)
print("maxRecordsPerFile=5 on 12 rows:", len(files_in(sized)), "files")
orders.repartition("status").write.mode("overwrite").partitionBy("status").parquet(sized)
print("repartition by the partition column:", files_in(sized))

### Tables: `saveAsTable` and `insertInto`

`saveAsTable(name)` writes the data and registers it in the catalog; later reads use `spark.table(name)` or SQL.
`insertInto(name)` appends to (or with `overwrite=True` replaces) an existing table, matching columns **by position**,
not by name.

**Notes:** `insertInto` with columns in a different order silently writes values into the wrong columns when the types
are compatible. Select the columns in the table's order first: `df.select(spark.table(name).columns)`. Managed tables
store data under `spark.sql.warehouse.dir`; dropping them deletes the data.

In [ ]:
spark.sql("DROP TABLE IF EXISTS order_status")
spark.createDataFrame([], "order_id int, status string, note string").write.format("parquet").saveAsTable("order_status")
swapped = spark.createDataFrame([("paid", 1, "x")], "status string, order_id int, note string")
with expected_error():
    swapped.write.insertInto("order_status")  # by position: 'paid' would go into order_id
swapped.select(spark.table("order_status").columns).write.insertInto("order_status")
spark.table("order_status").show()

### Bucketing: `bucketBy` and `sortBy`

Bucketing hashes rows by key into a fixed number of files per partition and records that layout in the table's
metadata. Joins and aggregations on the bucket key between tables bucketed the same way can skip the shuffle.

**Use it for:** very large tables joined repeatedly on the same key.

**Notes:** only works with `saveAsTable` (the catalog stores the bucketing), and both sides need the same number of
buckets on the join key. On Delta tables use liquid clustering or Z-ordering instead.

In [ ]:
for t in ["orders_b", "items_b"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
orders.write.format("parquet").bucketBy(4, "order_id").sortBy("order_id").saveAsTable("orders_b")
order_items.write.format("parquet").bucketBy(4, "order_id").sortBy("order_id").saveAsTable("items_b")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force a sort-merge join so the shuffles are visible
bucketed = spark.table("orders_b").join(spark.table("items_b"), "order_id")
plain = orders.join(order_items, "order_id")
print("shuffles bucketed:", sum("Exchange" in l for l in plan_lines(bucketed)), "| plain:", sum("Exchange" in l for l in plan_lines(plain)))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

## Delta Lake

### Writing and reading Delta tables, history and time travel

Delta Lake stores Parquet files plus a **transaction log** (`_delta_log/`) that records every commit. That log gives
ACID writes (readers never see half a write), table history, time travel, `MERGE`, and schema enforcement.

| operation | code |
| --- | --- |
| write | `df.write.format("delta").mode(...).save(path)` or `.saveAsTable(name)` |
| read | `spark.read.format("delta").load(path)` / `spark.table(name)` |
| history | `DeltaTable.forPath(spark, path).history()` |
| time travel | `.option("versionAsOf", n)` or `.option("timestampAsOf", "2026-01-05")` |

**Notes:** time travel only reaches versions whose files `VACUUM` has not removed (7 days by default).

In [ ]:
from delta.tables import DeltaTable

delta_path = f"{WORK_DIR}/delta_orders"
orders.write.format("delta").mode("overwrite").save(delta_path)
orders.where("status = 'paid'").write.format("delta").mode("overwrite").save(delta_path)
DeltaTable.forPath(spark, delta_path).history().select("version", "operation").orderBy("version").show()
print("now:", spark.read.format("delta").load(delta_path).count(), "rows | version 0:", spark.read.format("delta").option("versionAsOf", 0).load(delta_path).count(), "rows")

### `MERGE`: upserts with `DeltaTable.merge`

`target.alias("t").merge(source.alias("s"), condition)` matches source rows to target rows, then applies clauses in
order:

| clause | effect |
| --- | --- |
| `.whenMatchedUpdate(condition=None, set={...})` / `.whenMatchedUpdateAll()` | update matched target rows |
| `.whenMatchedDelete(condition=None)` | delete matched target rows |
| `.whenNotMatchedInsert(values={...})` / `.whenNotMatchedInsertAll()` | insert source rows with no match |
| `.whenNotMatchedBySourceDelete()` / `...Update(...)` | act on target rows the source no longer has |
| `.execute()` | run it |

**Use it for:** CDC upserts, SCD1 and SCD2 dimensions, deduplicated incremental loads.

**Notes:** the source must have **at most one row per key**, or the merge fails (`DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE`);
deduplicate first. Add a condition on `updated_at` so an older change never overwrites a newer one.

In [ ]:
target_path = f"{WORK_DIR}/delta_customers"
customers.select("customer_id", "name", "region").write.format("delta").mode("overwrite").save(target_path)
changes = spark.createDataFrame([(4, "Delta Grocers", "South"), (9, "India Imports", "North")], "customer_id int, name string, region string")
target = DeltaTable.forPath(spark, target_path)
(target.alias("t").merge(changes.alias("s"), "t.customer_id = s.customer_id")
 .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
spark.read.format("delta").load(target_path).where("customer_id IN (4, 9)").orderBy("customer_id").show()
target.history().select("version", "operation", "operationMetrics.numTargetRowsUpdated", "operationMetrics.numTargetRowsInserted").orderBy("version").show(truncate=False)

### Replace part of a table: `replaceWhere`

`.mode("overwrite").option("replaceWhere", predicate)` atomically replaces only the rows matching the predicate,
and fails if the new data contains rows outside it. It is Delta's idempotent way to reload a day or a partition.

In [ ]:
by_day = f"{WORK_DIR}/delta_daily"
daily.write.format("delta").mode("overwrite").partitionBy("day").save(by_day)
(daily.where("day = '2026-01-05'").withColumn("status", F.lit("audited"))
 .write.format("delta").mode("overwrite").option("replaceWhere", "day = '2026-01-05'").save(by_day))
spark.read.format("delta").load(by_day).groupBy("day", "status").count().orderBy("day", "status").show(4)

### Schema enforcement and evolution

Delta rejects an append whose schema does not match the table (**enforcement**). To add new columns on purpose, pass
`.option("mergeSchema", "true")` (**evolution**); to replace the schema with an overwrite, `.option("overwriteSchema", "true")`.

**Notes:** enforcement is a feature: it stops an upstream change from silently corrupting a table. Evolve schemas
deliberately and review type changes.

In [ ]:
wider = orders.limit(1).withColumn("channel", F.lit("web"))
with expected_error():
    wider.write.format("delta").mode("append").save(delta_path)
wider.write.format("delta").mode("append").option("mergeSchema", "true").save(delta_path)
print(spark.read.format("delta").load(delta_path).columns)

### Maintenance: `OPTIMIZE`, `ZORDER BY` and `VACUUM`

| command | effect |
| --- | --- |
| `OPTIMIZE table` | compacts small files into larger ones |
| `OPTIMIZE table ZORDER BY (col)` | also co-locates rows with similar values of `col`, so filters skip more files |
| `VACUUM table RETAIN 168 HOURS` | deletes files no longer referenced by recent versions (ends time travel to older versions) |

**Notes:** schedule `OPTIMIZE` for tables written in small batches (streaming, frequent appends). Newer Delta versions
offer liquid clustering (`CLUSTER BY`) as a replacement for partitioning plus Z-order.

In [ ]:
for i in range(4):
    orders.where(f"order_id % 4 = {i}").write.format("delta").mode("append").save(f"{WORK_DIR}/delta_small")
print("files before:", len(files_in(f"{WORK_DIR}/delta_small")))
spark.sql(f"OPTIMIZE delta.`{WORK_DIR}/delta_small` ZORDER BY (customer_id)").select("metrics.numFilesAdded", "metrics.numFilesRemoved").show()

**Next:** practise these entries in [PySpark practice 10: reading and writing data](pyspark-practice-10-reading-and-writing.ipynb).